In [ ]:
import open3d as o3d
import numpy as np
import math
from scipy.spatial.transform import Rotation as R


def default_visualization(object, zoom=1.0):
    azimuth_deg = -45
    elevation_deg = -135

    az = math.radians(azimuth_deg)
    el = math.radians(elevation_deg)

    # spherical → cartesian (camera front vector)
    front = np.array([
    math.cos(el) * math.cos(az),
    math.cos(el) * math.sin(az),
    math.sin(el)])

    # Open3D camera looks toward object
    front = -front
    lookat = [0, 0, 0]
    up = [0, 0, 1]  # typical CAD Z-up

    o3d.visualization.draw_geometries(
        object,
        window_name="Default Visualization",
        width=1024,
        height=768,
        lookat=lookat,
        up=up,
        front=front,
        zoom=zoom
    )

def get_euler_from_matrix(R):
    """Extracts Roll, Pitch, Yaw from a 3x3 Rotation Matrix"""
    # Using the standard XYZ convention
    pitch = math.asin(-R[2, 0])
    roll = math.atan2(R[2, 1], R[2, 2])
    yaw = math.atan2(R[1, 0], R[0, 0])
    
    # Return as degrees for easier reading
    return [math.degrees(roll), math.degrees(pitch), math.degrees(yaw)]

def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees
    
    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T

def inspect_viewpoint(target_idx, target_points_downsampled, viewpoints_6d, viewpoints_visualization, mesh=None):
    target_point_coords = np.asarray(target_points_downsampled.points[target_idx])

    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.25)
    sphere.translate(target_point_coords)
    sphere.paint_uniform_color([1, 0, 0])

    selected_camera = viewpoints_visualization[target_idx]

    cam_pos = np.array(viewpoints_6d[target_idx][:3])

    line_points = [target_point_coords, cam_pos]
    line_indices = [[0, 1]]

    connector_line = o3d.geometry.LineSet(
        points=o3d.utility.Vector3dVector(line_points),
        lines=o3d.utility.Vector2iVector(line_indices)
    )
    connector_line.paint_uniform_color([0, 1, 0])

    print(f"\n--- Inspecting Viewpoint Index: {target_idx} ---")
    print(f"Target Surface Point: {target_point_coords}")
    print(f"Camera Position (6D): {viewpoints_6d[target_idx]}")

    return sphere, selected_camera, connector_line, cam_pos, target_point_coords

def camera_pov_visualization(
        geoms,
        camera_pose,
        show_world_frame=False,
        show_camera_frame=False,
        save=0,
        index=0):

    vis = o3d.visualization.Visualizer()
    if save:
        vis.create_window(visible=False)
    else:
        vis.create_window()

    if show_world_frame:
        world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50)
        vis.add_geometry(world_frame)

    for g in geoms:
        vis.add_geometry(g)

    ctr = vis.get_view_control()

    # --- Extract camera pose ---
    cam_pos = camera_pose[:3, 3]
    R_cam = camera_pose[:3, :3]

    # Optional camera frame visualization
    if show_camera_frame:
        cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=5)
        cam_frame.rotate(R_cam, center=(0, 0, 0))
        cam_frame.translate(cam_pos)
        vis.add_geometry(cam_frame)

    # --- Build extrinsic (world → camera) ---
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = R_cam.T
    extrinsic[:3, 3] = -R_cam.T @ cam_pos

    param = ctr.convert_to_pinhole_camera_parameters()
    param.extrinsic = extrinsic
    ctr.convert_from_pinhole_camera_parameters(param)

    vis.poll_events()
    vis.update_renderer()

    if save == 1:
        vis.capture_screen_image(f"viewpoints_candidate/viewpoint_object_{index}.png")
    elif save == 2:
        vis.capture_screen_image(f"viewpoints_candidate/viewpoint_pcd_{index}.png")
    else:
        vis.run()

    vis.destroy_window()

def hidden_point_removal_from_camera(pcd, camera_pose, radius_scale=5):
    """
    Returns visible-point cloud using Open3D HPR.
    Uses full 4x4 camera pose.
    """

    cam_pos = camera_pose[:3, 3]

    center = pcd.get_center()
    d_cam = np.linalg.norm(cam_pos - center)
    radius = d_cam * radius_scale

    _, pt_map = pcd.hidden_point_removal(cam_pos, radius)

    visible_pcd = pcd.select_by_index(pt_map)
    visible_pcd.paint_uniform_color([0, 1, 0])

    return visible_pcd



In [4]:
# Visualization setup

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model

visualization = []

mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better
# R_mat = mesh.get_rotation_matrix_from_axis_angle([0, 0, np.radians(90)])
# mesh.rotate(R_mat, center=[0, 0, 0])

pcd_all = o3d.geometry.PointCloud()
pcd_all = mesh.sample_points_poisson_disk(number_of_points=5000)
pcd_all.paint_uniform_color([0.25, 0, 0])
# visualization.append(pcd_all)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50, origin=[0, 0, 0])
visualization.append(world_frame)


default_visualization([mesh], zoom=1.0)

In [ ]:
# Check the visualization of one viewpoint in detail, including the camera position, target point, and the line connecting them.

check_all = False
index = 7

if not check_all:
    sphere, selected_camera, connector_line, cam_pos, target_point_coords = inspect_viewpoint(
        index,
        target_points_reduced,
        viewpoints_6d_reduced,
        viewpoints_visualization_reduced,
        mesh
    )

    camera_pose = pose_to_matrix(viewpoints_6d_reduced[index])

    default_visualization(
        [mesh, selected_camera, sphere, connector_line, world_frame],
        zoom=1.25
    )

    # Camera POV view (NEW — pose based)
    camera_pov_visualization(
        [mesh, sphere, connector_line],
        camera_pose,
        show_world_frame=False,
        show_camera_frame=True
    )

    np.set_printoptions(
        precision=6,
        suppress=True,
        formatter={'all': lambda x: f'{x:0.6f}'}
    )

    formatted_matrix = np.array2string(
        camera_pose,
        separator=', ',
        precision=6,
        suppress_small=True
    )

    print(formatted_matrix)

else:

    for index in range(len(viewpoints_6d_reduced)):

        sphere, selected_camera, connector_line, cam_pos, target_point_coords = inspect_viewpoint(
            index,
            target_points_reduced,
            viewpoints_6d_reduced,
            viewpoints_visualization_reduced,
            mesh
        )

        camera_pose = pose_to_matrix(viewpoints_6d_reduced[index])

        camera_pov_visualization(
            [mesh, sphere, connector_line],
            camera_pose,
            show_world_frame=False,
            show_camera_frame=False
        )


In [ ]:
# Remove hidden points from the camera's perspective using Open3D's Hidden Point Removal (HPR) algorithm

# Extract camera position from camera_pose
cam_pos = camera_pose[:3, 3]

# Compute radius based on camera position
radius_scale=5
center = pcd_all.get_center()
d_cam = np.linalg.norm(cam_pos - center)
radius = d_cam * radius_scale

# Perform Hidden Point Removal (HPR) based on the camera position
visible_pcd = hidden_point_removal_from_camera(pcd_all, camera_pose, radius)

# Visualize the result
camera_pov_visualization(
    [visible_pcd, sphere],
    camera_pose,
    index=index
)


In [ ]:
# Generate viewpoints for all points and save them to a file
print("Final feasible viewpoints count:", len(viewpoints_6d_reduced))

for i in range(len(viewpoints_6d_reduced)):

    # --- Build camera pose ---
    camera_pose = pose_to_matrix(viewpoints_6d_reduced[i])
    cam_pos = camera_pose[:3, 3]   # only needed for radius computation

    # --- Get target sphere (for visualization only) ---
    sphere, _, _, _, _ = inspect_viewpoint(
        i,
        target_points_downsampled,
        viewpoints_6d_reduced,
        viewpoints_visualization_reduced,
        mesh
    )

    # Save simulated IMAGE
    camera_pov_visualization(
        [mesh, sphere],
        camera_pose,
        save=1,
        index=i
    )

    # Save simulated PCD
    radius_scale = 5
    center = pcd_all.get_center()
    d_cam = np.linalg.norm(cam_pos - center)
    radius = d_cam * radius_scale

    # Pass the full camera pose to the HPR function
    visible_pcd = hidden_point_removal_from_camera(
        pcd_all,
        camera_pose,     # Pass the full camera pose instead of just cam_pos
        radius
    )

    camera_pov_visualization(
        [visible_pcd, sphere],
        camera_pose,
        save=2,
        index=i
    )

    o3d.io.write_point_cloud(
        f"viewpoints_candidate/viewpoint_simulated_{i}.pcd",
        visible_pcd
    )

    # ==============================
    # Save 4x4 pose
    # ==============================
    np.save(
        f"viewpoints_candidate/viewpoint_pose_{i}.npy",
        camera_pose
    )

# Example check
output = np.load("viewpoints_candidate/viewpoint_pose_0.npy")
print("\nExample of output file transformation:\n", output)